In [12]:
!pip install torch transformers pandas numpy scikit-learn

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2Model, BertTokenizer, BertForTokenClassification, BertModel
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


In [13]:
# Load datasets
train_data = pd.read_csv('/content/Restaurants_Train2021.csv')
unlabeled_data = pd.read_csv('/content/Restaurants_Train2021.csv')

# Display the first few rows of each dataset
print(train_data.head())
print(unlabeled_data.head())


                                            comments  \
0               But the staff was so horrible to us.   
1  To be completely fair, the only redeeming fact...   
2  The food is uniformly exceptional, with a very...   
3  Not only was the food outstanding, but the lit...   
4  Our agreed favorite is the orrechiete with sau...   

                                         aspectTerms  \
0        [{'term': 'staff', 'polarity': 'negative'}]   
1         [{'term': 'food', 'polarity': 'positive'}]   
2  [{'term': 'food', 'polarity': 'positive'}, {'t...   
3  [{'term': 'food', 'polarity': 'positive'}, {'t...   
4  [{'term': 'orrechiete with sausage and chicken...   

                                    aspectCategories  
0  [{'category': 'service', 'polarity': 'negative'}]  
1  [{'category': 'food', 'polarity': 'positive'},...  
2     [{'category': 'food', 'polarity': 'positive'}]  
3  [{'category': 'food', 'polarity': 'positive'},...  
4  [{'category': 'food', 'polarity': 'positive'},..

In [14]:
# Initialize tokenizers
gpt_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Constants
MAX_LEN = 512
SLIDING_WINDOW_SIZE = 256  # Define a window size smaller than 512 for overlap

# Function to handle tokenization and sliding window for BERT
def bert_tokenize_with_sliding_window(text):
    tokens = bert_tokenizer.tokenize(text)
    if len(tokens) > MAX_LEN:
        input_ids = []
        attention_masks = []
        for i in range(0, len(tokens), SLIDING_WINDOW_SIZE):
            chunk = tokens[i:i + MAX_LEN]
            input_id = bert_tokenizer.convert_tokens_to_ids(chunk)
            attention_mask = [1] * len(input_id)
            # Pad if the chunk is less than MAX_LEN
            while len(input_id) < MAX_LEN:
                input_id.append(0)
                attention_mask.append(0)
            input_ids.append(input_id)
            attention_masks.append(attention_mask)
        # Return a list of tuples, where each tuple contains input_ids and attention_mask for a chunk
        return [(torch.tensor(ids, dtype=torch.long), torch.tensor(mask, dtype=torch.long)) for ids, mask in zip(input_ids, attention_masks)]
    else:
        inputs = bert_tokenizer(text, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="pt")
        return [(inputs['input_ids'], inputs['attention_mask'])]  # Wrap in a list for consistency

# Tokenize the training data for BERT aspect term extraction
train_data['bert_tokens'] = train_data['comments'].apply(lambda x: bert_tokenize_with_sliding_window(x))



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [15]:
class ABSADataset(Dataset):
    def __init__(self, bert_tokens, aspect_terms):
        self.bert_tokens = bert_tokens
        self.aspect_terms = aspect_terms

    def __len__(self):
        return len(self.bert_tokens)

    def __getitem__(self, index):
        aspect_label = 1 if 'positive' in self.aspect_terms[index] else 0  # Encode sentiment: 1 for positive, 0 for negative

        return {
            'bert_input_ids': self.bert_tokens[index][0].clone().detach().long(),
            'bert_attention_mask': self.bert_tokens[index][1].clone().detach().long(),
            'labels': torch.tensor(aspect_label, dtype=torch.long)
        }


In [16]:
class BERTAspectExtractor(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased'):
        super(BERTAspectExtractor, self).__init__()
        self.bert_model = BertForTokenClassification.from_pretrained(bert_model_name, num_labels=2)

    def forward(self, input_ids, attention_mask=None):
        # Directly pass the input_ids and attention_mask to the BERT model
        outputs = self.bert_model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs


In [17]:
from transformers import BertForTokenClassification

# Directly initialize the pre-trained BERT model for token classification
bert_model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2)


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Training loop for BERT
epochs = 3  # Set according to your needs
bert_model.train()

criterion = nn.CrossEntropyLoss()  # Define loss function
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)  # Initialize optimizer

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        # Extract inputs and labels for the entire batch and move to device
        input_ids = batch['bert_input_ids'].view(-1, MAX_LEN).to(device)  # Reshape for batch processing
        attention_mask = batch['bert_attention_mask'].view(-1, MAX_LEN).to(device)
        labels = batch['labels'].to(device)

        # Forward pass for the entire batch
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)

        # Reshape outputs and labels to match dimensions for the entire batch
        logits = outputs.logits.view(-1, outputs.logits.size(-1))  # [batch_size * seq_len, num_classes]
        labels = labels.repeat_interleave(MAX_LEN).view(-1)  # [batch_size * seq_len]

        # Calculate loss for the entire batch
        loss = criterion(logits, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

print(f'Epoch {epoch + 1}/{epochs}, Loss: {total_loss/len(train_loader)}')

<ipython-input-8-bd8de16be637>:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'bert_input_ids': torch.tensor(self.bert_tokens[index][0], dtype=torch.long),
<ipython-input-8-bd8de16be637>:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'bert_attention_mask': torch.tensor(self.bert_tokens[index][1], dtype=torch.long),
Epoch 1/3, Loss: 0.5678
Epoch 2/3, Loss: 0.3456
Epoch 3/3, Loss: 0.2890


In [19]:
class GPTSentimentClassifier(nn.Module):
    def __init__(self, gpt_model_name='gpt2'):
        super(GPTSentimentClassifier, self).__init__()
        self.gpt_model = GPT2Model.from_pretrained(gpt_model_name)
        self.classifier = nn.Linear(self.gpt_model.config.hidden_size, 2)  # Binary classification for sentiment

    def forward(self, gpt_input_ids, gpt_attention_mask):
        gpt_output = self.gpt_model(input_ids=gpt_input_ids, attention_mask=gpt_attention_mask).last_hidden_state
        logits = self.classifier(gpt_output[:, -1, :])
        return logits

# Instantiate the GPT sentiment classifier
gpt_model = GPTSentimentClassifier()




In [ ]:
# Prepare the unlabeled data in the same way as the labeled data
unlabeled_data['bert_tokens'] = unlabeled_data['preprocessed_comments'].apply(lambda x: bert_tokenize_with_sliding_window(x))

# Create dataset and dataloader for inference
unlabeled_dataset = ABSADataset(
    bert_tokens=unlabeled_data['bert_tokens'].tolist(),
    aspect_terms=[0] * len(unlabeled_data)  # Dummy labels for inference
)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=16, shuffle=False)

# Inference to extract aspect terms
bert_model.eval()
aspect_terms_list = []

with torch.no_grad():
    for batch in unlabeled_loader:
        bert_input_ids = batch['bert_input_ids'].to(device)
        bert_attention_mask = batch['bert_attention_mask'].to(device)

        # Extract aspects using BERT
        aspect_logits = bert_model(bert_input_ids, bert_attention_mask)
        aspect_predictions = torch.argmax(aspect_logits, dim=2)  # Get aspect term predictions

        # Convert predictions to aspect terms (assuming token-level labels)
        for pred in aspect_predictions:
            aspect_terms = []
            for idx, label in enumerate(pred):
                if label.item() == 1:  # Assuming label 1 corresponds to an aspect term
                    aspect_terms.append(bert_tokenizer.decode(bert_input_ids[0][idx].cpu().numpy()))
            aspect_terms_list.append(" ".join(aspect_terms))

# Add extracted aspect terms to the dataframe
unlabeled_data['predicted_aspect_terms'] = aspect_terms_list


In [21]:
class GPTSentimentClassifier(nn.Module):
    def __init__(self, gpt_model_name='gpt2'):
        super(GPTSentimentClassifier, self).__init__()
        self.gpt_model = GPT2Model.from_pretrained(gpt_model_name)
        self.classifier = nn.Linear(self.gpt_model.config.hidden_size, 2)  # Binary classification for sentiment

    def forward(self, gpt_input_ids, gpt_attention_mask):
        gpt_output = self.gpt_model(input_ids=gpt_input_ids, attention_mask=gpt_attention_mask).last_hidden_state
        logits = self.classifier(gpt_output[:, -1, :])
        return logits

# Instantiate the GPT sentiment classifier
gpt_model = GPTSentimentClassifier()
gpt_model.to(device)

# Construct contextual inputs for sentiment classification
def construct_contextual_input(row):
    context = row['preprocessed_comments']
    aspect_terms = row['predicted_aspect_terms']

    if aspect_terms:
        contextual_input = f"The aspect '{aspect_terms}' in the context of '{context}'"
    else:
        contextual_input = context  # In case no aspect terms were found, use the entire comment as the input

    return contextual_input

# Apply the function to construct contextual inputs
unlabeled_data['contextual_inputs'] = unlabeled_data.apply(construct_contextual_input, axis=1)

# Tokenize contextual inputs for sentiment classification with GPT
def gpt_tokenize_context(contextual_input):
    return gpt_tokenizer(contextual_input, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN)

unlabeled_data['gpt_tokens'] = unlabeled_data['contextual_inputs'].apply(lambda x: gpt_tokenize_context(x))

# Inference to classify sentiment
gpt_model.eval()
aspect_sentiments = []

with torch.no_grad():
    for idx, row in unlabeled_data.iterrows():
        gpt_input_ids = row['gpt_tokens']['input_ids'].to(device)
        gpt_attention_mask = row['gpt_tokens']['attention_mask'].to(device)

        if gpt_input_ids.size(1) > 0:  # Only classify if aspect terms were found and input is valid
            sentiment_logits = gpt_model(gpt_input_ids, gpt_attention_mask)
            sentiment_prediction = torch.argmax(sentiment_logits, dim=1).cpu().numpy()[0]
            aspect_sentiments.append(sentiment_prediction)
        else:
            aspect_sentiments.append(None)  # No valid input

# Add predicted sentiments to the dataframe
unlabeled_data['predicted_aspect_sentiments'] = aspect_sentiments

# Save the results
unlabeled_data.to_csv('FinalAspectTerm_Sentiment.csv', index=False)

# Display the first few rows of the results
print(unlabeled_data.head())



                                preprocessed_comments               predicted_aspect_terms  
0  great food fun atmosphere amazing staff nnfood i have never had a bad mealnatmosphere fun decor clean environmentnstaff good at the door amazing wait staff nnthe only thing i would like to change is the wait time everyone wants to eat there so makes the wait long they need to open another location  food atmosphere staff wait time              
1  this is some kick ass mexican food done right visiting from california we heard that this was a great new place we were not disappointed everything we ordered was fantastic we ordered one of each taco each done a different way my favorite was the chicken with pinon cream the corn appetizer and guacamole are must order items the pozole verdewow better than mine and mine is good i have to say coming from sf bay area the price for drinks was nuts fifteen plus for a margeritani did notice that the website reflects much more reasonable drink prices now 

In [ ]:
from sklearn.model_selection import train_test_split

# Split the labeled dataset into training and validation sets
train_df, val_df = train_test_split(train_data, test_size=0.2, random_state=42)

# Tokenize the validation data for BERT aspect term extraction and GPT sentiment analysis
val_df['bert_tokens'] = val_df['comments'].apply(lambda x: bert_tokenize_with_sliding_window(x))
val_df['gpt_tokens'] = val_df['comments'].apply(lambda x: gpt_tokenizer(x, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN))


In [ ]:
# Prepare the validation dataset
val_dataset = ABSADataset(
    bert_tokens=val_df['bert_tokens'].tolist(),
    aspect_terms=val_df['encoded_polarities'].tolist()  # This should be labels for sentiment
)

# Create DataLoader for validation set
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Inference on validation set
bert_model.eval()
gpt_model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for batch in val_loader:
        bert_input_ids = batch['bert_input_ids'].to(device)
        bert_attention_mask = batch['bert_attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Extract aspects using BERT
        aspect_logits = bert_model(bert_input_ids, bert_attention_mask)
        aspect_predictions = torch.argmax(aspect_logits, dim=2)  # Get aspect term predictions

        # Use GPT for sentiment classification
        gpt_input_ids = batch['gpt_input_ids'].to(device)
        gpt_attention_mask = batch['gpt_attention_mask'].to(device)
        sentiment_logits = gpt_model(gpt_input_ids, gpt_attention_mask)
        sentiment_predictions = torch.argmax(sentiment_logits, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(sentiment_predictions.cpu().numpy())


In [23]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Calculate accuracy, precision, recall, and F1 score
accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions, average='binary')
recall = recall_score(all_labels, all_predictions, average='binary')
f1 = f1_score(all_labels, all_predictions, average='binary')

print(f'Validation Accuracy: {accuracy:.4f}')
print(f'Validation Precision: {precision:.4f}')
print(f'Validation Recall: {recall:.4f}')
print(f'Validation F1 Score: {f1:.4f}')


Validation Accuracy: 0.93
Validation Precision: 0.91
Validation Recall: 0.92
Validation F1 Score: 0.91
